In [5]:
# peforming sminrov k-s test for all flux sampling reactions 
import os
import pandas as pd
from scipy import stats

In [7]:
input_path = '/Users/eso1993/Library/CloudStorage/Box-Box/CP2_project/Flux_sampling' # input for flux sampling 
print(os.listdir(input_path))

['iMAT_Veh-NTG4_sampling.csv', 'iMAT_Veh-NTG3_sampling.csv', 'iMAT_Veh-NTG2_sampling.csv', 'iMAT_Veh-NTG5_sampling.csv', 'iMAT_CP2-APPPS1-4_sampling.csv', 'iMAT_CP2-APPPS1-3_sampling.csv', 'iMAT_Veh-APPPS1-5_sampling.csv', 'iMAT_Veh-APPPS1-2_sampling.csv', 'iMAT_Veh-APPPS1-3_sampling.csv', 'iMAT_Veh-APPPS1-4_sampling.csv', 'iMAT_CP2-APPPS1-2_sampling.csv', 'iMAT_CP2-APPPS1-5_sampling.csv', 'iMAT_Veh-APPPS1-1_sampling.csv', 'iMAT_CP2-APPPS1-1_sampling.csv', 'iMAT_Veh-NTG1_sampling.csv']


In [9]:
# Step 1: Define paths and group structure ===

groups = {
    "NTG-Veh": [f for f in os.listdir(input_path) if "Veh-NTG" in f],
    "APPPS1-Veh": [f for f in os.listdir(input_path) if "Veh-APPPS1" in f],
    "APPPS1-CP2": [f for f in os.listdir(input_path) if "CP2-APPPS1" in f]
}

# Load all flux sampling data for each group
flux_data = {}
for group, files in groups.items():
    dfs = []
    for f in files:
        df = pd.read_csv(os.path.join(input_path, f))
        dfs.append(df)
    flux_data[group] = pd.concat(dfs, axis=0, ignore_index=True)

print({g: d.shape for g, d in flux_data.items()})  # sanity check


{'NTG-Veh': (5000, 4549), 'APPPS1-Veh': (5000, 4700), 'APPPS1-CP2': (5000, 4760)}


In [11]:
# Step 2: Align reactions across all groups
common_reactions = set.intersection(*[set(df.columns) for df in flux_data.values()])
for g in flux_data:
    flux_data[g] = flux_data[g][list(common_reactions)]

# Step 3: Run pairwise KS tests for each reaction ===
comparisons = [
    ("APPPS1-Veh", "NTG-Veh"),
    ("APPPS1-CP2", "NTG-Veh"),
    ("APPPS1-CP2", "APPPS1-Veh")
]

results = []

for rxn in common_reactions:
    for g1, g2 in comparisons:
        try:
            ks_stat, pval = stats.ks_2samp(flux_data[g1][rxn], flux_data[g2][rxn])
            results.append({
                "Reaction": rxn,
                "Group1": g1,
                "Group2": g2,
                "KS_Statistic": ks_stat,
                "P_value": pval
            })
        except Exception as e:
            print(f"Skipping {rxn} ({g1} vs {g2}): {e}")
            continue

# Step 4: Save output 
results_df = pd.DataFrame(results)
output_file = os.path.join(input_path, "/Users/eso1993/Desktop/Flux_KS_results.csv")
results_df.to_csv(output_file, index=False)